#Comparacion Modelo de red LSTM y simpleRNN pequeño para generar texto

In [ ]:
import tensorflow as tf
print(tf.__version__)

# Importo librerias
import sys
import numpy
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Dropout
from keras.layers import LSTM, SimpleRNN, GRU
#from keras.utils import np_utils

from tensorflow.keras.utils import to_categorical
#
# Carga del archivo de texto y convertirlo en minúscula
filename = "los3.txt"
raw_text = open(filename).read()
raw_text = raw_text.lower()

# Crea un diccionario de caracteres a enteros y viceversa
chars = sorted(list(set(raw_text)))
char_to_int = dict((c, i) for i, c in enumerate(chars))
int_to_char = dict((i, c) for i, c in enumerate(chars))

# Contabiliza los datos
n_chars = len(raw_text)
n_vocab = len(chars)
print("Total Characters: ", n_chars)
print("Total Vocab: ", n_vocab)

# Prepara el conjunto de datos de pares de entrada a salida codificados
seq_length = 100
dataX = []
dataY = []
for i in range(0, n_chars - seq_length, 1):
    seq_in = raw_text[i:i + seq_length]
    seq_out = raw_text[i + seq_length]
    dataX.append([char_to_int[char] for char in seq_in])
    dataY.append(char_to_int[seq_out])
    n_patterns = len(dataX)
print("Total Patterns: ", n_patterns)

# Da formato a la X
X = numpy.reshape(dataX, (n_patterns, seq_length, 1))
# Normalización
X = X / float(n_vocab)
# Codificación de la variable de salida
y = to_categorical(dataY)


2.18.0
Total Characters:  3140
Total Vocab:  38
Total Patterns:  3040


In [ ]:
# Define el modelo GRU
modelgr = Sequential()
modelgr.add(GRU(256, input_shape=(X.shape[1], X.shape[2])))
modelgr.add(Dropout(0.2))
modelgr.add(Dense(y.shape[1], activation='softmax'))

#Compila el modelo SimpleGRU
modelgr.compile(loss='categorical_crossentropy', optimizer='adam')

/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
#Entrena modelo GRU
modelgr.fit(X, y, epochs=3000, batch_size=64)

Se han truncado las últimas 5000 líneas del flujo de salida.
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0147
Epoch 502/3000
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0155
Epoch 503/3000
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0610
Epoch 504/3000
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.1900
Epoch 505/3000
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.1482
Epoch 506/3000
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0825
Epoch 507/3000
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0352
Epoch 508/3000
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0313
Epoch 509/3000
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0259
Epoch 510/3000
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0113
Epoch 511/3000
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0123
Epoch 512/3000
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0088
Epoch 513/3000
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0080
Epoch 514/3000
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step 

In [ ]:
# Toma una semilla aleatoria
start = numpy.random.randint(0, len(dataX)-1)
pattern = dataX[start]
print("Semilla:")
print(start)
print(pattern)
print("\"", ''.join([int_to_char[value] for value in pattern]), "\"")


Semilla:
176
[24, 1, 21, 23, 20, 21, 15, 7, 24, 1, 9, 7, 24, 7, 24, 1, 21, 7, 23, 7, 1, 11, 24, 25, 7, 23, 1, 21, 23, 20, 25, 11, 13, 15, 10, 20, 24, 5, 1, 7, 1, 17, 20, 24, 1, 20, 25, 23, 20, 24, 1, 10, 20, 24, 1, 17, 11, 24, 1, 21, 7, 23, 11, 9, 15, 36, 1, 26, 19, 7, 1, 8, 26, 11, 19, 7, 1, 15, 10, 11, 7, 3, 1, 29, 1, 24, 11, 1, 21, 26, 24, 15, 11, 23, 20, 19, 1, 18, 7, 19]
" s propias casas para estar protegidos. a los otros dos les pareció una buena idea, y se pusieron man "


In [ ]:
#Generacion de caracteres con GRU
print("Texto")
for i in range(300):
  x = numpy.reshape(pattern, (1, len(pattern), 1))
  x = x / float(n_vocab)
  prediction = modelgr.predict(x, verbose=0)
  index = numpy.argmax(prediction)
  result = int_to_char[index]
  seq_in = [int_to_char[value] for value in pattern]
  sys.stdout.write(result)
  pattern.append(index)
  pattern = pattern[1:len(pattern)]


print("\nHecho.")

Texto
oe a la ourré de ma abra ne maror  lrreó pr p r parepro  r besiero y ror al leciaio. dalor r qoalor tre lor eer arors sr cás s po logvt eo qmeia slríó ll eáaodes que eereiros rer hreedas ve ponvigddos ann c carirto . ses  s res p ros eeroi  - ¡aerdito a ror  rer cesdito soe so  as nabca y eeoe  o so
Hecho.


Crear un pattern distinto y ver lo q genera

In [ ]:
semilla_personalizada = "facilidad"

pattern = [char_to_int[char] for char in semilla_personalizada]

while len(pattern) < 100:
    pattern.insert(0, 0)

In [ ]:
#Generacion de caracteres con GRU
print("Texto")
for i in range(300):
  x = numpy.reshape(pattern, (1, len(pattern), 1))
  x = x / float(n_vocab)
  prediction = modelgr.predict(x, verbose=0)
  index = numpy.argmax(prediction)
  result = int_to_char[index]
  seq_in = [int_to_char[value] for value in pattern]
  sys.stdout.write(result)
  pattern.append(index)
  pattern = pattern[1:len(pattern)]

print("\nHecho.")

Texto
dstdd dentaad  oesedes quq oueod  l lo los eerdieoses eeredba centina  - ¡quién teme al lobo feroz, al lobo,  l lob lo  le los elaede alnada de paeo, los bor eey iayo sos tereato y hoteabe laloi la pesoielo ay an  a ¡ierrer  elrerao yos leroinoe yo coa le limo eesaz   h ladacd  all deedtt l lo rasod
Hecho.
